# Financial NLP Intelligence: Naive Bayes vs Logistic Regression vs SVM

**Apeiron AI — Boundless Possibilities, Infinite Potential**

## Objective
Build an end-to-end NLP pipeline using the FiQA-2018 financial dataset:
- Text preprocessing
- Tokenization
- TF-IDF representation
- Model comparison
- Evaluation
- Save best model for deployment

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os

from datasets import load_dataset
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import *

nltk.download("punkt")
nltk.download("stopwords")

In [ ]:
# Set a global seed for reproducibility
import random

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

## 2. Load Dataset

In [ ]:

dataset=load_dataset("pauri32/fiqa-2018")

train_df=pd.DataFrame(dataset['train'])
val_df=pd.DataFrame(dataset['validation'])
test_df=pd.DataFrame(dataset['test'])

print(train_df.head())
print(train_df.columns)


In [ ]:
TEXT_COL = 'sentence'
LABEL_COL = 'label'

## 3. Dataset Exploration

In [ ]:

print(train_df.shape)

train_df.head()


### Sentiment Label Distribution

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x=LABEL_COL, data=train_df, palette='viridis')
plt.title('Distribution of Sentiment Labels in Training Data')
plt.xlabel('Sentiment Label')
plt.ylabel('Count')
plt.show()

## 4. Text Preprocessing

### Further Data Exploration: Sentence Length and Word Frequency

In [ ]:
# Calculate sentence length
train_df['sentence_length'] = train_df[TEXT_COL].apply(lambda x: len(str(x).split()))
val_df['sentence_length'] = val_df[TEXT_COL].apply(lambda x: len(str(x).split()))
test_df['sentence_length'] = test_df[TEXT_COL].apply(lambda x: len(str(x).split()))

# Visualize sentence length distribution
plt.figure(figsize=(10, 6))
sns.histplot(train_df['sentence_length'], bins=50, kde=True)
plt.title('Distribution of Sentence Lengths in Training Data')
plt.xlabel('Sentence Length (words)')
plt.ylabel('Frequency')
plt.show()


Let's analyze word frequencies, which can highlight key terms in financial sentiment. We'll do this by sentiment label.

In [ ]:
from collections import Counter

# Ensure 'clean_text' column exists before proceeding
# This handles cases where preprocessing might not have been run or was run out of order
if 'clean_text' not in train_df.columns:
    # Assuming 'preprocess' function is defined in an earlier cell (a249a6b6) and available
    train_df["clean_text"] = train_df[TEXT_COL].apply(preprocess)

def get_top_n_words(corpus, n=10):
    words = []
    for text in corpus:
        for word in text.split():
            words.append(word)
    return Counter(words).most_common(n)

# Group by sentiment label and get top words
for label_val in sorted(train_df[LABEL_COL].unique()):
    subset_corpus = train_df[train_df[LABEL_COL] == label_val]['clean_text']
    top_words = get_top_n_words(subset_corpus, n=15)
    print(f"\nTop 15 words for Label {label_val}:")
    for word, count in top_words:
        print(f"  {word}: {count}")


### Addressing Class Imbalance with Class Weights

Given the imbalance observed in the sentiment label distribution (especially for Label 1 in the classification report), we can calculate class weights to assign higher penalties to misclassifications of underrepresented classes. This helps the models pay more attention to minority classes during training.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Calculate class weights for the training data
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to a dictionary for use in models
class_weights = dict(zip(np.unique(y_train), class_weights_array))

print("Calculated Class Weights:")
print(class_weights)

### Retraining Models with Class Weights

Now that we have calculated class weights, we can retrain our models, passing these weights to the model constructors. This will help the models prioritize the underrepresented classes during training, potentially improving performance on the minority class (Label 1).

In [ ]:
print("Models with Class Weights:")

models_weighted = {
    "NaiveBayes_Weighted": MultinomialNB(), # MultinomialNB does not directly support class_weight in constructor
    "LogisticRegression_Weighted": LogisticRegression(max_iter=1000, class_weight=class_weights, random_state=RANDOM_SEED),
    "SVM_Weighted": SVC(probability=True, class_weight=class_weights, random_state=RANDOM_SEED)
}

In [ ]:
results_weighted = []
trained_models_weighted = {}

print("Retraining models with class weights...")

for name, model in models_weighted.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    preds_weighted = model.predict(X_test)

    acc_weighted = accuracy_score(y_test, preds_weighted)
    precision_weighted = precision_score(y_test, preds_weighted, average='weighted', zero_division=0)
    recall_weighted = recall_score(y_test, preds_weighted, average='weighted', zero_division=0)
    f1_weighted = f1_score(y_test, preds_weighted, average='weighted', zero_division=0)

    results_weighted.append(
        [name, acc_weighted, precision_weighted, recall_weighted, f1_weighted]
    )

    trained_models_weighted[name] = model

print("Retraining complete.")

### Results with Class Weights

Let's examine the performance of the models after training with class weights.

In [ ]:
results_df_weighted = pd.DataFrame(
    results_weighted,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1"]
)

display(results_df_weighted)

### Classification Report for `LogisticRegression_Weighted`

Let's get a detailed classification report for the `LogisticRegression_Weighted` model to understand its performance on each class with the applied class weights.

In [ ]:
from sklearn.metrics import classification_report

# Get the LogisticRegression_Weighted model
logreg_weighted_model = trained_models_weighted["LogisticRegression_Weighted"]

# Generate predictions for this specific model on the test set
preds_logreg_weighted = logreg_weighted_model.predict(X_test)

# Print the classification report
print(classification_report(y_test, preds_logreg_weighted, zero_division=0))

### Classification Report for `SVM_Weighted`

Let's get a detailed classification report for the `SVM_Weighted` model to understand its performance on each class with the applied class weights.

In [ ]:
from sklearn.metrics import classification_report

# Get the SVM_Weighted model
svm_weighted_model = trained_models_weighted["SVM_Weighted"]

# Generate predictions for this specific model on the test set
preds_svm_weighted = svm_weighted_model.predict(X_test)

# Print the classification report
print(classification_report(y_test, preds_svm_weighted, zero_division=0))

### F1-Score Comparison of All Models

To get a clear overview of which models perform best, let's compare the F1-scores of all trained models, including both the unweighted and weighted versions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Combine the results from both dataframes
all_results_df = pd.concat([results_df, results_df_weighted], ignore_index=True)

plt.figure(figsize=(12, 7))
sns.barplot(x='Model', y='F1', data=all_results_df, palette='viridis')
plt.title('F1-Score Comparison Across All Models')
plt.xlabel('Model')
plt.ylabel('F1-Score')
plt.ylim(0, 1) # F1-score is between 0 and 1
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:

stemmer=PorterStemmer()
stop_words=set(stopwords.words('english'))

def preprocess(text):

    text=str(text).lower()
    text=re.sub(r'[^a-zA-Z ]','',text)

    words=word_tokenize(text)

    words=[
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)


## 5. Apply Preprocessing

In [ ]:
import nltk
nltk.download('punkt_tab')

TEXT_COL='sentence'

train_df["clean_text"]=train_df[TEXT_COL].apply(preprocess)
val_df["clean_text"]=val_df[TEXT_COL].apply(preprocess)
test_df["clean_text"]=test_df[TEXT_COL].apply(preprocess)


## 6. TF-IDF Representation

In [ ]:

vectorizer=TfidfVectorizer(max_features=5000)

X_train=vectorizer.fit_transform(train_df["clean_text"])
X_test=vectorizer.transform(test_df["clean_text"])

LABEL_COL='label'

y_train=train_df[LABEL_COL]
y_test=test_df[LABEL_COL]


## 7. Define Models

In [ ]:

models={

"NaiveBayes":MultinomialNB(),

"LogisticRegression":
LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),

"SVM":
SVC(probability=True, random_state=RANDOM_SEED)

}

## 8. Train Models

In [ ]:

results=[]
trained_models={}

for name,model in models.items():

    model.fit(X_train,y_train)

    preds=model.predict(X_test)

    acc=accuracy_score(y_test,preds)
    precision=precision_score(y_test,preds,average='weighted')
    recall=recall_score(y_test,preds,average='weighted')
    f1=f1_score(y_test,preds,average='weighted')

    results.append(
        [name,acc,precision,recall,f1]
    )

    trained_models[name]=model


## 9. Results

In [ ]:

results_df=pd.DataFrame(
results,
columns=[
"Model",
"Accuracy",
"Precision",
"Recall",
"F1"
]
)

results_df


## 10. Confusion Matrix

In [ ]:

best_model_name=results_df.sort_values(
'F1',
ascending=False
).iloc[0]["Model"]

best_model=trained_models[best_model_name]

preds=best_model.predict(X_test)

cm=confusion_matrix(y_test,preds)

plt.figure(figsize=(8,5))
sns.heatmap(cm,annot=True,fmt='d')
plt.show()


### Classification Report

To get a more detailed evaluation of the best model, we can generate a classification report. This report will show the precision, recall, and F1-score for each class, as well as overall averages.

In [ ]:
from sklearn.metrics import classification_report

# Generate predictions for the best model on the test set
preds = best_model.predict(X_test)

# Print the classification report
print(classification_report(y_test, preds))

## 11. Save Model

In [ ]:
model_dir = "../model/"
os.makedirs(model_dir, exist_ok=True)

pickle.dump(
    best_model,
    open(f"{model_dir}best_model.pkl","wb")
)

pickle.dump(
    vectorizer,
    open(f"{model_dir}tfidf_vectorizer.pkl","wb")
)

with open(
    f"{model_dir}config.json",
    "w"
) as f:

    json.dump({
        "model":best_model_name
    },f,indent=2)

print("Saved")

## 12. Summary

In [ ]:

print(results_df)
